# Chat Completions Providers

The `chat_completions` provider talks to any OpenAI-compatible Chat Completions endpoint.

Register an `AsyncOpenAI` client under `"chat_completions"` with a `name`, then select models with the `name/` prefix (the remainder is the API model id, and may itself contain `/`). One router holds as many endpoints as you like, so this notebook registers all of them on a single `router`.


In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import EasyInputMessageParam

from interop_router.router import Router
from interop_router.types import ChatMessage, RouterResponse

load_dotenv(find_dotenv())


def extract_text(response: RouterResponse) -> str:
    texts: list[str] = []
    for chat_message in response.output:
        if chat_message.message.get("type") == "message":
            content = chat_message.message.get("content")
            if isinstance(content, list):
                for c in content:
                    text = c.get("text", "")
                    if text:
                        texts.append(text)
    return "\n".join(texts)


message = ChatMessage(message=EasyInputMessageParam(role="user", content="Hello! Reply in one short sentence."))

router = Router()

## OpenAI

Use a default `AsyncOpenAI()` client (reads `OPENAI_API_KEY`). The name `openai` is reserved for the Responses provider, so this endpoint is registered as `openai_chat`. The prefix is what sends the request through Chat Completions instead of the Responses API.


In [2]:
router.register("chat_completions", AsyncOpenAI(), name="openai_chat")

response = await router.create(
    input=[message],
    model="openai_chat/gpt-5.6-luna",
)
print(extract_text(response))

Hello! How can I help you today?


## Local vLLM

Point `base_url` at a local OpenAI-compatible server. Set `CHAT_COMPLETIONS_BASE_URL` and `CHAT_COMPLETIONS_MODEL` in `.env` (see `.env.example`). The model id after the name is passed through unchanged.


In [3]:
base_url = os.environ["CHAT_COMPLETIONS_BASE_URL"]
model = os.environ["CHAT_COMPLETIONS_MODEL"]

router.register("chat_completions", AsyncOpenAI(base_url=base_url), name="vllm")

response = await router.create(
    input=[message],
    model=f"vllm/{model}",
)
print(extract_text(response))



Hello, how can I help you today?


## OpenRouter

OpenRouter exposes an OpenAI-compatible Chat Completions API. Set `base_url` and provide an `OPENROUTER_API_KEY`.


In [4]:
router.register(
    "chat_completions",
    AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    ),
    name="openrouter",
)

response = await router.create(
    input=[message],
    model="openrouter/poolside/laguna-xs-2.1:free",
)
print(extract_text(response))

Hello! How can I assist you today?


## OpenCode Zen

[OpenCode Zen](https://opencode.ai/docs/zen/)

In [5]:
router.register(
    "chat_completions",
    AsyncOpenAI(
        base_url="https://opencode.ai/zen/v1",
        api_key=os.getenv("OPENCODE_API_KEY"),
    ),
    name="opencode",
)

response = await router.create(
    input=[message],
    model="opencode/deepseek-v4-flash-free",
)
print(extract_text(response))

Hello! How can I help you today?


## Selecting an endpoint

All four endpoints now live on one router. A name selects one of them directly. The bare `chat_completions/` prefix cannot, because it no longer identifies a single client.


In [6]:
try:
    await router.create(input=[message], model="chat_completions/gpt-5.6-luna")
except ValueError as e:
    print(e)

# A name that would shadow a provider is rejected at registration time.
try:
    router.register("chat_completions", AsyncOpenAI(), name="openai")
except ValueError as e:
    print(e)

# Unknown ids are never routed to a Chat Completions endpoint by default.
try:
    await router.create(input=[message], model="nvidia/Qwen3.6-27B-NVFP4")
except ValueError as e:
    print(e)

Multiple clients are registered for provider 'chat_completions': openai_chat, opencode, openrouter, vllm. Use a 'name/model' reference to select one.
Registration name 'openai' is reserved for the provider of the same name.
Unknown model: 'nvidia/Qwen3.6-27B-NVFP4'. Use a supported bare model id, or a 'provider/model' reference (providers: anthropic, chat_completions, gemini, openai).


## Token counting

Chat Completions has no native count endpoint. InteropRouter estimates prompt tokens with tiktoken using OpenAI Chat Completions framing (not arbitrary chat templates such as Llama on vLLM).

The model id after the name is treated as a tiktoken encoding name (`o200k_base`, `cl100k_base`, ...) or an OpenAI model id that tiktoken knows. Unknown values fall back to `o200k_base`. Counting is local; no API call is made, so any registered endpoint gives the same answer.


In [7]:
from openai.types.responses.function_tool_param import FunctionToolParam

get_weather = FunctionToolParam(
    type="function",
    name="get_weather",
    description="Get the current weather in a location",
    parameters={
        "type": "object",
        "properties": {"location": {"type": "string"}},
        "required": ["location"],
    },
    strict=True,
)

base = await router.count_tokens(
    input=[message],
    model="openai_chat/gpt-5.6-luna",
)
with_instructions = await router.count_tokens(
    input=[message],
    model="openai_chat/nvidia/Qwen3.6-27B-NVFP4",
    instructions="Always answer in one short sentence.",
)
with_tools = await router.count_tokens(
    input=[message],
    model="openai_chat/nvidia/Qwen3.6-27B-NVFP4",
    tools=[get_weather],
)
# Prefer an explicit encoding when the API model id is unknown to tiktoken.
with_encoding = await router.count_tokens(
    input=[message],
    model="openai_chat/o200k_base",
)

print(f"base: {base}")
print(f"with instructions: {with_instructions}")
print(f"with tools: {with_tools}")
print(f"explicit encoding: {with_encoding}")

base: 15
with instructions: 26
with tools: 54
explicit encoding: 15
